# Connect and authorize google drive

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')
!ls

In [ ]:
# %cd /content/gdrive/My Drive/
%cd /content/gdrive/My Drive/Colab Notebooks/ML_HI/tano_signal/
!ls

# Libraries

In [ ]:
import os
import sys

from astropy.io import fits
from astropy    import units as u

import numpy as np
import pandas as pd
import math
import copy

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.optim.lr_scheduler import LambdaLR, StepLR, MultiStepLR, ExponentialLR
from torch.utils.data import Dataset, DataLoader


from pathlib import Path

# GPU or CPU

In [ ]:
# GPU or CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device: ', device)
print('device_count: ', torch.cuda.device_count())

print('Torch version: ', torch.__version__)
print('torch.version.cuda: ', torch.version.cuda)

print("Is CUDA enabled?",torch.cuda.is_available())

device:  cpu
device_count:  0
Torch version:  2.4.1+cu121
torch.version.cuda:  12.1
Is CUDA enabled? False


# Paths

In [ ]:
base_path = Path('/content/gdrive/MyDrive/Colab Notebooks/ML/')

tano_signal_path = base_path / 'tano_signal' / 'tano_signal'
checkpoint_path  = tano_signal_path / 'checkpoints'
ct101_checkpoint_path = checkpoint_path / 'c101'
cnn101_checkpoint_path = checkpoint_path / 'cnn' / 'c101'

data_path     = base_path / 'data'
uma_data_path = data_path / 'UMA'

tanosignal_path  = base_path / 'tano_signal'
tanosignal_pred_path = tanosignal_path / 'pred'

saury2014_data_path = data_path / 'saury'
seta222_data_path = data_path / 'seta'

In [ ]:
# Training and test datasets
# 1. Datacubes: 04 datacubes with size of (512 x 512 x Nchan)
training_data_file_q0 = 'Tb_n01_pw02_vs12_512_thick_256chan_quarter_0_noise_1.0_K_beam_1.45_pix.fits'
training_data_file_q1 = 'Tb_n01_pw02_vs12_512_thick_256chan_quarter_1_noise_1.0_K_beam_1.45_pix.fits'
training_data_file_q2 = 'Tb_n01_pw02_vs12_512_thick_256chan_quarter_2_noise_1.0_K_beam_1.45_pix.fits'
training_data_file_q3 = 'Tb_n01_pw02_vs12_512_thick_256chan_quarter_3_noise_1.0_K_beam_1.45_pix.fits'

training_data_file_q0 = saury2014_data_path / training_data_file_q0
training_data_file_q1 = saury2014_data_path / training_data_file_q1
training_data_file_q2 = saury2014_data_path / training_data_file_q2
training_data_file_q3 = saury2014_data_path / training_data_file_q3

# HI optical depth
tau_data_file_q0 = 'tau_n01_pw02_vs12_512_thick_256chan_quarter_0_noise_beam_1.45_pix.fits'
tau_data_file_q1 = 'tau_n01_pw02_vs12_512_thick_256chan_quarter_1_noise_beam_1.45_pix.fits'
tau_data_file_q2 = 'tau_n01_pw02_vs12_512_thick_256chan_quarter_2_noise_beam_1.45_pix.fits'
tau_data_file_q3 = 'tau_n01_pw02_vs12_512_thick_256chan_quarter_3_noise_beam_1.45_pix.fits'

tau_data_file_q0 = saury2014_data_path / tau_data_file_q0
tau_data_file_q1 = saury2014_data_path / tau_data_file_q1
tau_data_file_q2 = saury2014_data_path / tau_data_file_q2
tau_data_file_q3 = saury2014_data_path / tau_data_file_q3


# 2a. R_HI maps
rhi_file_q0 = 'R_map_n01_pw02_vs12_512px_quarter_0_noise.fits'
rhi_file_q1 = 'R_map_n01_pw02_vs12_512px_quarter_1_noise.fits'
rhi_file_q2 = 'R_map_n01_pw02_vs12_512px_quarter_2_noise.fits'
rhi_file_q3 = 'R_map_n01_pw02_vs12_512px_quarter_3_noise.fits'

rhi_file_q0 = saury2014_data_path / rhi_file_q0
rhi_file_q1 = saury2014_data_path / rhi_file_q1
rhi_file_q2 = saury2014_data_path / rhi_file_q2
rhi_file_q3 = saury2014_data_path / rhi_file_q3

# 3a. FCNM maps with noise
fcnm_file_q0 = 'fcnm_map_n01_pw02_vs12_Tcut_500K_512px_quarter_0_noise.fits'
fcnm_file_q1 = 'fcnm_map_n01_pw02_vs12_Tcut_500K_512px_quarter_1_noise.fits'
fcnm_file_q2 = 'fcnm_map_n01_pw02_vs12_Tcut_500K_512px_quarter_2_noise.fits'
fcnm_file_q3 = 'fcnm_map_n01_pw02_vs12_Tcut_500K_512px_quarter_3_noise.fits'

fcnm_file_q0 = saury2014_data_path / fcnm_file_q0
fcnm_file_q1 = saury2014_data_path / fcnm_file_q1
fcnm_file_q2 = saury2014_data_path / fcnm_file_q2
fcnm_file_q3 = saury2014_data_path / fcnm_file_q3

print(training_data_file_q0)
print(fcnm_file_q0)
print(rhi_file_q0)

In [ ]:
training_data_file_q4 = 'Tb_mpism_sol_800_thick_256chan.fits'
training_data_file_q4 = seta222_data_path / training_data_file_q4

tau_data_file_q4 = 'tau_mpism_sol_800_turb_thick_256chan.fits'
tau_data_file_q4 = seta222_data_path / tau_data_file_q4

rhi_file_q4 = 'rhi_map_mpism_sol_800_Tcut_500K_thick_256chan.fits'
rhi_file_q4 = seta222_data_path / rhi_file_q4

fcnm_file_q4 = 'fcnm_map_mpism_sol_800_Tcut_500K.fits'
fcnm_file_q4 = seta222_data_path / fcnm_file_q4


print(training_data_file_q4)
print(fcnm_file_q4)
print(rhi_file_q4)

# Fcn

In [ ]:
def save_csv(trial_valid_err, trial_test_err, trial_filename, process_train_err, process_validate_err, process_test_err, training_process_filename):
    """
        save_csv: save CSV files.

        input Attributes
        ----------
        trial_valid_err : list/array
            the validation performance in each trail.
        trial_test_err: list/array
            the testing performance in each trail.
        trial_filename: String
            csv file name for storing the testing and validation data.


        process_train_err : list/array
            train err of each epoch.
        process_validate_err : list/array
            validating err of each epoch.
        process_test_err : list/array
            testing err of each epoch.
        training_process_filename:  String
            csv file name for storing the process data.
    """
    ntrials = len(trial_valid_err)
    nepochs = len(process_train_err) // ntrials

    # Create an array containing numbers from 0 to nepochs
    numbers = np.arange(nepochs)

    # Repeat the array 10 times
    epochs = np.tile(numbers, ntrials)

    df = pd.DataFrame({'valid_err' : trial_valid_err, 'test_err' : trial_test_err})
    df.to_csv(trial_filename, index=False)

    #
    df = pd.DataFrame({'epoch' : epochs, 'train_err' : process_train_err, 'valid_err' : process_validate_err, 'test_err' : process_test_err})
    df.to_csv(training_process_filename, index=False)

# Preprocessing

In [ ]:
# data loader
class SpecDataset(Dataset):
    def __init__(self, x, y, pev, xtransform=None, ytransform=None):
        self.x = x
        self.y = y
        self.xtransform = xtransform
        self.ytransform = ytransform

        self.pev = pev
        self.input_column = input_column

        self.get_pe()



    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        x = self.x[:, idx] # shape: (nchan, nbatch)
        y = self.y[idx]    # shape: (2)

        # Positional encoding
        x = self.pe_fcn(x)

        if self.xtransform:
            x = self.xtransform(np.asarray(x, dtype = 'float32'))

        if self.ytransform:
            y = self.ytransform(np.asarray(y, dtype = 'float32'))

        return x, y




    def pe_index_concate(self, spec):
        pos  = np.linspace(0, 1., self.input_column).reshape(1, -1)
        spec = np.vstack((spec, pos))
        spec = spec.reshape(1, 2, self.input_column)
        return spec

    def pe_index_add(self, spec):
        pos   = np.linspace(0, 1., self.input_column).reshape(1, -1)
        spec  = spec + pos
        spec  = spec.reshape(1, 1, self.input_column)
        return spec

    def pe_sin_add(self, spec):
        pos  = np.sin(np.linspace(0, 1., self.input_column).reshape(1, -1))
        spec = spec + pos
        spec = spec.reshape(1, 1, self.input_column)
        return spec

    def pe_sin_concate(self, spec):
        pos  = np.sin(np.linspace(0, 1., self.input_column).reshape(1, -1))
        spec = np.vstack((spec, pos))
        spec = spec.reshape(1, 2, self.input_column)
        return spec

    def pe_poly_concate(self, spec):
        spec = np.vstack((spec, spec**2))
        spec = spec.reshape(1, 2, self.input_column)
        return spec


    def pe_original_vector(self, spec):
        return spec.reshape(1, 1, self.input_column)


    def get_pe(self):
        pes = {
            'index_concate'   : self.pe_index_concate,
            'index_add'       : self.pe_index_add,
            'sin_concate'     : self.pe_sin_concate,
            'sin_add'         : self.pe_sin_add,
            'poly_concate'    : self.pe_poly_concate,
            'original_vector' : self.pe_original_vector,
            'trainable_add'   : self.pe_original_vector
        }

        self.pe_fcn = pes.get(self.pev)


# Add transformations here
class ToTensor():
    def __call__(self, sample):
        x = torch.from_numpy(sample)
        return x

In [ ]:
"""
# Created on 01 September 2023
# @author: Van Hiep Nguyen
"""

"""
    ToDict: A class of dictionary maker.
    generate a dictionary for the datacube that conatins ground truth.
    Used for 101 velocity channels data (data with 101 length).

    initialization Attributes
    ----------
    cube: data cube
        spectral data cube.
    Rhi: cube
        Ground-truth RHI
    Fcnm:  cube
        Ground-truth FCNM

    Methods
    -------
    make_dict_gt:
        generate a dictionary for the datacube that contains ground truth.
        it can be used for training and prediction.

    make_dict:
        generate a dictionary for observed datacube
        it can be used for prediction only.

"""

from typing import List, Optional

class Prep:
    def __init__(self,
                 pev: str,
                 input_column: int,
                 batch_size: int,
                 cubes: List[np.ndarray],
                 fcnm: Optional[List[np.ndarray]] = None,
                 rhi: Optional[List[np.ndarray]] = None) -> np.ndarray:

        self.pev   = pev
        self.batch_size = batch_size


        if (rhi is not None) and (fcnm is not None):
            nc = len(cubes)
            nf = len(fcnm)
            nr = len(rhi)
            if (nc != nf) or (nc != nr):
                return None



            self.trainmode = True
            self.n = nc

        if (rhi is None) or (fcnm is None):
            self.trainmode = False # Prediction
            self.n = len(cubes)

        self.nchan, self.ny, self.nx = cubes[0].shape
        nyf, nxf = fcnm[0].shape
        nyr, nxr = rhi[0].shape
        self.nspec = self.nx * self.ny * self.n
        self.cube_nspec = self.nx * self.ny


        # Add more validations here, should write a function: validate()
        if (self.ny != nyf) or (self.ny != nyr):
            print('Data sizes do not match!')
            print(self.ny, nyf, nyr)
            print(self.nx, nxf, nxr)
            return None

        # reshape the cubes and maps
        self.xtrainset = np.zeros((self.nchan, self.nspec), dtype='float32')
        self.ytrainset = np.zeros((self.nspec, 2)) # 2 values in label (fcnm, rhi)

        print('xtrain length: ', self.nspec)

        for k, (cube, ymap, zmap) in enumerate(zip(cubes, fcnm, rhi)):
            cube = cube.reshape(self.nchan, -1, order='C')
            self.xtrainset[:, (k*self.cube_nspec) : (k+1)*self.cube_nspec] = cube[:, 0:self.cube_nspec]
            self.ytrainset[(k*self.cube_nspec) : (k+1)*self.cube_nspec, 0] = ymap.reshape(self.cube_nspec, order='C')
            self.ytrainset[(k*self.cube_nspec) : (k+1)*self.cube_nspec, 1] = zmap.reshape(self.cube_nspec, order='C')


    def fit(self):
        if self.trainmode:
            print('Training mode...')
            return self.xtrainset, self.ytrainset

        self.predfit()
        nchan, nrow, ncol = self.cubes[0].shape
        num = nrow * ncol

        print('Data cube shape: ', nchan, nrow, ncol)
        recube = self.cubes.reshape(nchan, -1, order='C')
        print('Reshaped cube: ', recube.shape)

        if (self.rhi is None) or (self.fcnm is None):
            print('Prediction mode ...')
            return recube

        if (self.rhi is not None) and (self.fcnm is not None):
            print('Training mode ...')
            return recube, self.fcnm.reshape(num, order='C'), self.rhi.reshape(num, order='C')

        return None

# Models

In [ ]:
# -*- coding: utf-8 -*-
"""
@author: Haiyang.Tang
Do not used for commercial purpose!

Transformer Positional Encoder

    PositionalEncoding: positional encoding method in original natural language processing.

    Methods
    -------
    forward(x):
        add positional encoding to x and return x.
"""


class PE(nn.Module):
    def __init__(self, num_features, sequence_len=6, d_model=9):
        super(PE, self).__init__()

        pe     = torch.zeros((1, sequence_len, d_model), dtype=torch.float32)
        factor = -math.log(10000.) / d_model  # outs loop

        for index in range(0, sequence_len):  # position of word in seq
            for i in range(0, d_model, 2):
                div_term = math.exp(i * factor)
                pe[0, index, i] = math.sin(index * div_term)
                if (i+1 < d_model):
                    pe[0, index, i+1] = math.cos(index * div_term)

        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:x.size(0), :]
        return x

In [ ]:
class Models(nn.Module):
    def __init__(self, pev, num_output, input_column, dropout, device):
        super(Models, self).__init__()

        self.pev = pev

        self.num_output   = num_output
        self.in_channels  = 1
        self.input_column = input_column
        self.dropout      = dropout
        self.device       = device

        self.get_input_row() # 1 or 2 (concate)
        self.get_lpe()
        # self.get_pe()

        self.loss_fcn = nn.MSELoss()



    def get_lpe(self):
        if self.pev == 'trainable_add':
            self.lpe = True
        else:
          self.lpe = False

    def get_input_row(self):
        if self.pev in ['index_concate', 'sin_concate', 'poly_concate']:
            self.input_row = 2
        else:
            self.input_row = 1






    def load_weights(self, checkpoint_path):
        """
        Load check point

        Parameters:


        Returns:
         -
        """

        print('Checkpoint file: ', checkpoint_path)

        try:
            if checkpoint_path.is_file():
                checkpoint = torch.load(checkpoint_path, map_location=self.device)
                self.load_state_dict(checkpoint['net'])
                self.to(self.device)
                self.eval()

                print('sucessfully load the weights!')
            else:
                print("The file does not exist.")
                print(checkpoint_path)
        except Exception as e:
            print("An error occurred while reading the file:", e)



    # Calc. output size
    def get_output_size(self, Hin, Win, k = [3, 3], s = [1, 1], p = [0, 0], d = [1, 1]):
        return int((Hin + 2*p[0] - d[0]*(k[0] - 1)/s[0] - 1) + 1), int((Win + 2*p[1] - d[1]*(k[1] - 1)/s[1] - 1) + 1)


    def predict(self, cube):
        nchan, nrow, ncol = cube.shape
        ntotal = nrow * ncol
        dat = Prep().fit(cube, fcnm=None, rhi=None)

        fcnm = np.zeros(ntotal)
        rhi  = np.zeros(ntotal)
        with torch.no_grad():
            for i in range(ntotal):
                if i%1e4 == 0:
                  print(i, '/', ntotal)

                spec = dat[:nchan, i]

                spec = self.pe_fcn(spec)
                spec = spec.reshape(1, spec.shape[0], spec.shape[1], -1).astype(np.float32)
                spec = torch.from_numpy(spec)
                spec = spec.to(self.device)

                # Feed single spectrum (+pe)
                pred = self(spec)

                fcnm[i] = pred[0][0].cpu().detach().numpy()
                rhi[i]  = pred[0][1].cpu().detach().numpy()
            # End - for
        # End - with

        fcnm = fcnm.reshape(nrow, ncol, order='C')
        rhi  = rhi.reshape(nrow, ncol, order='C')

        print('Complete !')
        return fcnm, rhi


    def epoch_train(self):
        train_loss = 0.
        for idx, (x, y) in enumerate(self.train_loader):
            x = x.float()
            y = y.float()
            x, y = x.to(device), y.to(device)

            # Zero your gradients for every batch!
            self.optimizer.zero_grad()

            # forward
            ret = self(x)

            # Compute the loss and its gradients
            loss = self.loss_fcn(ret, y)
            loss.backward()

            # Adjust weights
            self.optimizer.step()

            train_loss += loss.item()
        # End - for

        avgloss = train_loss / len(self.train_loader)
        return train_loss, avgloss






    def epoch_validate(self, epoch):
        """
        validate the model.
        Parameters
        ----------
        PEV: String
            positional encoding vector method, used for naming the weights file.
        epoch : int.
            the index of epoch.

        Returns
        -------
        error: The mean MSE in validation set.

        """

        valid_loss = 0.
        with torch.no_grad():
            for idx, (x, y) in enumerate(self.valid_loader):
                x = x.float()
                y = y.float()
                x, y = x.to(device), y.to(device)

                # forward
                ret = self(x)
                loss      = self.loss_fcn(ret, y)
                valid_loss += loss.item()
            # End - for
        # End - with

        # Save checkpoint.
        error =  valid_loss / len(self.valid_loader)
        if error < self.best_valid_loss:
            print('Best MSE err: ', error, 'Saving..')
            state = {'net': self.state_dict(),
                    'err': error,
                    'optimizer_state_dict': self.optimizer.state_dict(),
                    'epoch': epoch
                    }

            torch.save(state, checkpoint_path / f'{self.pev}.pth')
            self.best_valid_loss = error
        #End - if



    def test(self):
        """
        Test model.
        Parameters
        ----------
        PEV: String
            positional encoding vector method, used for naming the weights file.
        epoch : int.
            the index of epoch.

        Returns
        -------
        error: The mean MSE in test set.

        """

        test_loss = 0.
        with torch.no_grad():
            for idx, (x, y) in enumerate(self.test_loader):
                x = x.float()
                y = y.float()
                x, y = x.to(device), y.to(device)

                # forward
                ret = self(x)
                loss      = self.loss_fcn(ret, y)
                test_loss += loss.item()
            # End - for
        # End - with

        # Average loss
        avgloss =  test_loss / len(self.test_loader)
        return avgloss





    def fit(self, x, y, batch_size=200, lr=0.009, epochs=2):
        # prep = Prep(self.pev, self.input_column, batch_size, cubes, fcnm=fcnm, rhi=rhi)

        print('PE: ', self.pev)
        print('Trainable positional encoding: ', self.lpe)
        print('Learning rate: ', lr)
        print('Batchsize: ', batch_size)

        # To CPU/GPU
        self.to(self.device)

        self.optimizer = optim.SGD(self.parameters(), lr)
        self.scheduler = MultiStepLR(optimizer=self.optimizer, milestones=[48], gamma=0.1, last_epoch=-1, verbose=False)

        self.best_valid_loss       = 9_999_999.
        self.best_trial_valid_loss = 9_999_999.

        nchan, nspec = x.shape

        test_len = nspec//5
        idx_arr  = np.arange(nspec)

        np.random.shuffle(idx_arr)

        test_idx  = idx_arr[0 : test_len]
        valid_idx = idx_arr[test_len : 2*test_len]
        train_idx = idx_arr[2*test_len : nspec]

        xtrain = x[:, train_idx]
        ytrain = y[train_idx]

        xvalid = x[:, valid_idx]
        yvalid = y[valid_idx]

        xtest = x[:, test_idx]
        ytest = y[test_idx]

        print(x.shape)
        print(y.shape)

        print('Train length: ', xtrain.shape[1])
        print('Validation length: ', xvalid.shape[1])
        print('Test length: ', xtest.shape[1])
        print('All: ', xtrain.shape[1] + xvalid.shape[1] + xtest.shape[1])
        assert (xtrain.shape[1]+xvalid.shape[1]+xtest.shape[1] == nspec)


        dataset_train = SpecDataset(xtrain, ytrain, self.pev, xtransform=ToTensor(), ytransform=ToTensor())
        dataset_val   = SpecDataset(xtrain, ytrain, self.pev, xtransform=ToTensor(), ytransform=ToTensor())
        dataset_test  = SpecDataset(xtrain, ytrain, self.pev, xtransform=ToTensor(), ytransform=ToTensor())


        # initialize data loader (x, y)
        self.train_loader = DataLoader(dataset = dataset_train, batch_size = batch_size, shuffle = True)
        self.valid_loader = DataLoader(dataset = dataset_val,   batch_size = batch_size, shuffle = False)
        self.test_loader  = DataLoader(dataset = dataset_test,  batch_size = batch_size, shuffle = False)

        for (a,b) in self.train_loader:
            print(a.shape, b.shape)
            break


        in_channels = a.shape[1] # 1 concat
        input_row   = a.shape[2] # 2 concat
        print(in_channels, input_row)

        for epoch in range(epochs):
            print('-----> epoch: ', epoch+1, '/', epochs)
            self.train(True)
            train_loss, avgloss = self.epoch_train()
            print(f'Total loss in epoch {epoch} = {train_loss}, average loss = {avgloss}')

            # Set the model to evaluation mode, disabling dropout and using population
            # statistics for batch normalization.
            self.eval()
            self.epoch_validate(epoch)
        # End - for: epochs

            testloss = self.test()

        print()

# Child models

In [ ]:
"""
    cnn_transformer_small: A class of the CNN_transformer small model.

    input Attributes
    ----------
    num_features : int
        The number of features.
    drop_rate : the drop out rate in transformer.
        input channel number.
    pos_encoder : PositionalEncoding.
        original positional encoder in NLP.
    lpe: boolean
        whether or not add leanable positional embedding to transformer inputs.
    pos_embedding: nn.Parameter(.)
        learnable parameters for positional embedding.

    Methods
    -------
    forward(x):
        calculate the outputs.

"""
class CNN_Transformer(Models):
    def __init__(self, pev, num_output, input_column, dropout, device):
        super().__init__(pev, num_output, input_column, dropout, device)

        p = [0, 0] # padding
        d = [1, 1] # dilation
        k = [1, 6] # kernael_size
        s = [1, 1] # stride

        self.num_features = 54

        kernel_wid = 33 if self.input_column == 256 else 7

        self.pos_encoder   = PE(num_features=self.num_features, sequence_len=6, d_model=9)
        self.pos_embedding = nn.Parameter(torch.randn(self.in_channels,self.input_row, self.input_column))

        # CNN layers (outchannels = outchannels-8)
        kernelsize = (1,3) if (self.input_row < 2) else (2,3)
        self.conv1 = nn.Conv2d(in_channels=self.in_channels, out_channels=72, kernel_size=kernelsize, stride=1, padding=0, bias=True, padding_mode='zeros')
        self.bn1   = nn.BatchNorm2d(72)
        Hout, Wout = self.get_output_size(72, self.input_column, k=kernelsize, s=s, p=p, d=d)

        self.conv2 = nn.Conv2d(in_channels=72, out_channels=64, kernel_size=(1,kernel_wid), stride=1, padding=0, bias=True, padding_mode='zeros')
        self.bn2 = nn.BatchNorm2d(64)
        Hout, Wout = self.get_output_size(64, Wout, k=(1,kernel_wid), s=s, p=p, d=d)

        self.conv3 = nn.Conv2d(in_channels=64, out_channels=56, kernel_size=(1,3), stride=1, padding=0, bias=True, padding_mode='zeros')
        self.bn3 = nn.BatchNorm2d(56)
        Hout, Wout = self.get_output_size(56, Wout, k = (1,3), s=s, p=p, d=d)

        self.conv4 = nn.Conv2d(in_channels=56, out_channels=48,  kernel_size=(1,kernel_wid), stride=1, padding=0, bias=True, padding_mode='zeros')
        self.bn4 = nn.BatchNorm2d(48)
        Hout, Wout = self.get_output_size(48, Wout, k = (1,kernel_wid), s=s, p=p, d=d)

        self.conv5 = nn.Conv2d(in_channels=48, out_channels=40,  kernel_size=(1,3), stride=1, padding=0, bias=True, padding_mode='zeros')
        self.bn5 = nn.BatchNorm2d(40)
        Hout, Wout = self.get_output_size(40, Wout, k = (1,3), s=s, p=p, d=d)

        self.conv6 = nn.Conv2d(in_channels=40, out_channels=32,  kernel_size=(1,kernel_wid), stride=1, padding=0, bias=True, padding_mode='zeros')
        self.bn6 = nn.BatchNorm2d(32)
        Hout, Wout = self.get_output_size(32, Wout, k = (1,kernel_wid), s=s, p=p, d=d)

        self.conv7 = nn.Conv2d(in_channels=32, out_channels=16,  kernel_size=(1,3), stride=1, padding=0,  bias=True, padding_mode='zeros')
        self.bn7 = nn.BatchNorm2d(16)
        Hout, Wout = self.get_output_size(16, Wout, k = (1,3), s=s, p=p, d=d)

        self.conv8 = nn.Conv2d(in_channels=16, out_channels=8,  kernel_size=(1,kernel_wid), stride=1, padding=0, bias=True, padding_mode='zeros')
        self.bn8 = nn.BatchNorm2d(8)
        Hout, Wout = self.get_output_size(8, Wout, k = (1,kernel_wid), s=s, p=p, d=d)

        self.conv9 = nn.Conv2d(in_channels=8, out_channels=4,  kernel_size=(1,3), stride=1, padding=0, bias=True, padding_mode='zeros')
        self.bn9 = nn.BatchNorm2d(4)
        Hout, Wout = self.get_output_size(4, Wout, k = (1,3), s=s, p=p, d=d)

        self.conv10 = nn.Conv2d(in_channels=4, out_channels=2,  kernel_size=(1,kernel_wid), stride=1, padding=0, bias=True, padding_mode='zeros')
        self.bn10 = nn.BatchNorm2d(2)
        Hout, Wout = self.get_output_size(2, Wout, k = (1,kernel_wid), s=s, p=p, d=d)

        self.linear = nn.Linear(Hout*Wout, 54)
        self.flatten = nn.Flatten()

        self.transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(
                d_model=9,
                nhead=3,
                dim_feedforward=36,
                dropout=self.dropout,
                batch_first=True,
            ),
            num_layers=4
        )

        self.decoder = nn.Linear(54, self.num_output)

        # init parameter
        for m in self.modules():
            if isinstance(m, nn.Conv1d):
                n = m.kernel_size[0]*m.out_channels
                m.weight.data.normal_(0, math.sqrt(2./n))
            elif isinstance(m, nn.BatchNorm1d):
                m.weight.data.fill_(1)
                m.bias.data.zero_()
            elif isinstance(m, nn.Linear):
                m.bias.data.zero_()

    def forward(self, x):
        if self.lpe:
            x =  x + self.pos_embedding

        x = self.conv1(x)
        x = self.bn1(x)
        x = F.relu(x)

        x = self.conv2(x)
        x = self.bn2(x)
        x = F.relu(x)

        x = self.conv3(x)
        x = self.bn3(x)
        x = F.relu(x)

        x = self.conv4(x)
        x = self.bn4(x)
        x = F.relu(x)
        #
        x = self.conv5(x)
        x = self.bn5(x)
        x = F.relu(x)
        #
        x = self.conv6(x)
        x = self.bn6(x)
        x = F.relu(x)
        #
        x = self.conv7(x)
        x = self.bn7(x)
        x = F.relu(x)
        #
        x = self.conv8(x)
        x = self.bn8(x)
        x = F.relu(x)

        x = self.conv9(x)
        x = self.bn9(x)
        x = F.relu(x)

        x = self.conv10(x)
        x = self.bn10(x)
        x = F.relu(x)


        x = self.flatten(x)

        x = self.linear(x)

        x = x.reshape(x.shape[0], -1, 9)

        # Transformer MODEL
        x = self.transformer(x)
        x = self.flatten(x)

        x = self.decoder(x)
        return x












"""
    CNN: A class of the CNN model.

    input Attributes
    ----------
    drop_out_rate : int
        the drop out rate in last layer of CNN.
    lpe: boolean
        whether or not add leanable positional embedding to CNN inputs.
    num_layer : int
        number of layers of CNN.
    num_output: int
        number of output
    in_channels: int
        number of input channels
    input_row : int
        number of input row
    input_column: int
        number of input column

    Methods
    -------
    forward(x):
        calculate the outputs.

"""

class CNN(Models):
    def __init__(self, pev, num_output, input_column, dropout, device, num_layer):
        super().__init__(pev, num_output, input_column, dropout, device)

        kernel_wid_cvlayer1 = 7
        kernel_wid_cvlayer2 = 33
        if self.input_column == 101:
            kernel_wid_cvlayer2 = 7

        if self.input_column == 256:
            kernel_wid_cvlayer2 = 33

        p = [0, 0] # padding
        d = [1, 1] # dilation
        k = [1, 6] # kernael_size
        s = [1, 1] # stride

        self.num_layer    = num_layer
        self.num_features = self.in_channels*self.input_row*self.input_column

        self.out_channels_1 = self.num_layer*8 + 8
        self.dropout        = nn.Dropout(self.dropout)
        self.pos_embedding  = nn.Parameter(torch.randn(self.in_channels, self.input_row, self.input_column))

        # Weights
        self.weight_cp_file = cnn101_checkpoint_path / f'cnn_{self.pev}_{num_layer}_{self.input_column}_final.pth'

        # head of CNN
        kernel_row_conv1  = 1 if (self.input_row < 2) else 2
        kernel_size_conv1 = (kernel_row_conv1, kernel_wid_cvlayer1)
        self.conv1 = nn.Conv2d(in_channels = self.in_channels, out_channels=self.out_channels_1,
                              kernel_size=kernel_size_conv1, stride=1, padding=0,
                              bias=True, padding_mode='zeros')

        self.bn1 = nn.BatchNorm2d(self.out_channels_1)

        kernel_size_conv2 = (1, kernel_wid_cvlayer1)
        self.conv2 = nn.Conv2d(in_channels= self.out_channels_1, out_channels=self.out_channels_1-8,
                               kernel_size=kernel_size_conv2, stride=1, padding=0, bias=True,
                               padding_mode='zeros')
        self.bn2 = nn.BatchNorm2d(self.out_channels_1-8)



        # add layers
        self.layers = nn.ModuleList()
        self.layers.append(self.conv1)
        self.layers.append(self.bn1)
        self.layers.append(nn.ReLU())
        Hout, Wout = self.get_output_size(self.out_channels_1, self.input_column, k = kernel_size_conv1, s = s, p = p, d = d)

        self.layers.append(self.conv2)
        self.layers.append(self.bn2)
        self.layers.append(nn.ReLU())
        Hout, Wout = self.get_output_size(self.out_channels_1-8, Wout, k = kernel_size_conv2, s = s, p = p, d = d)

        count = self.out_channels_1-8
        if num_layer >= 4:
            for i in range(0, int((num_layer-4)/2)):
                #1
                self.layers.append(nn.Conv2d(in_channels=count, out_channels=count-8,
                               kernel_size=(1,6), stride=1, padding=0, bias=True,
                               padding_mode='zeros'))
                self.layers.append(nn.BatchNorm2d(count-8))
                self.layers.append(nn.ReLU())
                Hout, Wout = self.get_output_size(count-8, Wout, k = (1,6), s = s, p = p, d = d)
                # 2
                self.layers.append(nn.Conv2d(in_channels= count-8, out_channels=count-16,
                               kernel_size=(1,kernel_wid_cvlayer2), stride=1, padding=0, bias=True,
                               padding_mode='zeros'))
                self.layers.append(nn.BatchNorm2d(count-16))
                self.layers.append(nn.ReLU())
                Hout, Wout = self.get_output_size(count-16, Wout, k = (1,kernel_wid_cvlayer2), s = s, p = p, d = d)
                count = count-16
            # End - for

            self.layers.append(nn.Conv2d(in_channels = count, out_channels=16,
                               kernel_size=(1,6), stride=1, padding=0,
                               bias=True, padding_mode='zeros'))
            self.layers.append(nn.BatchNorm2d(16))
            self.layers.append(nn.ReLU())
            Hout, Wout = self.get_output_size(16, Wout, k = (1,6), s = s, p = p, d = d)

            self.layers.append(nn.Conv2d(in_channels= 16, out_channels=8,
                               kernel_size=(1,kernel_wid_cvlayer2), stride=1,
                               padding=0, bias=True, padding_mode='zeros'))
            self.layers.append(nn.BatchNorm2d(8))
            self.layers.append(nn.ReLU())
            Hout, Wout = self.get_output_size(8, Wout, k = (1,kernel_wid_cvlayer2), s = s, p = p, d = d)
        # End - if

        ###
        self.layers.append(nn.Flatten())
        self.layers.append(self.dropout)

        # FC layer
        self.linear = nn.Linear(Hout*Wout, self.num_output)

        # init parameter
        for m in self.modules():
            if isinstance(m, nn.Conv1d):
                n = m.kernel_size[0]*m.out_channels
                m.weight.data.normal_(0, math.sqrt(2. / n))
            elif isinstance(m, nn.BatchNorm1d):
                m.weight.data.fill_(1)
                m.bias.data.zero_()
            elif isinstance(m, nn.Linear):
                m.bias.data.zero_()

    # Forward
    def forward(self, x):
        if self.lpe:
            x = x + self.pos_embedding

        for k, layer in enumerate(self.layers[:-1]):
            print('Layer: ', k)
            x = layer(x)
        # Endfor

        return self.linear(x)

# Read data

In [ ]:
dv = 0.3125
print('dv:', dv)

dv: 0.3125


In [ ]:
cube1, header1 = fits.getdata(training_data_file_q0, header=True)
cube2, header2 = fits.getdata(training_data_file_q1, header=True)
cube3, header3 = fits.getdata(training_data_file_q2, header=True)
cube4, header4 = fits.getdata(training_data_file_q3, header=True)
#
fcnm_data1, header1 = fits.getdata(fcnm_file_q0, header=True)
fcnm_data2, header2 = fits.getdata(fcnm_file_q1, header=True)
fcnm_data3, header3 = fits.getdata(fcnm_file_q2, header=True)
fcnm_data4, header4 = fits.getdata(fcnm_file_q3, header=True)
#
rhi_data1, header1 = fits.getdata(rhi_file_q0, header=True)
rhi_data2, header2 = fits.getdata(rhi_file_q1, header=True)
rhi_data3, header3 = fits.getdata(rhi_file_q2, header=True)
rhi_data4, header4 = fits.getdata(rhi_file_q3, header=True)

print('Saury2014 data cubes: ')
nchan, ny, nx = cube1.shape
print('nchan, ny, nx: ', nchan, ny, nx)
print(fcnm_data1.shape)
print(rhi_data1.shape)
print()


# Seta22
cube5, header5      = fits.getdata(training_data_file_q4, header=True)
fcnm_data5, header5 = fits.getdata(fcnm_file_q4, header=True)
rhi_data5, header5  = fits.getdata(rhi_file_q4, header=True)

print('Seta2022 data cubes: ')
nchan, ny, nx = cube5.shape
print('nchan, ny, nx: ', nchan, ny, nx)
print(fcnm_data5.shape)
print(rhi_data5.shape)

Saury2014 data cubes: 
nchan, ny, nx:  256 512 512
(512, 512)
(512, 512)

Seta2022 data cubes: 
nchan, ny, nx:  256 512 512
(512, 512)
(512, 512)


# Load models

In [ ]:
pev_list = ['original_vector', 'index_add', 'index_concate', 'poly_concate', 'sin_add', 'sin_concate']

In [ ]:
# CNN-Transformer model
pev = 'sin_add'
input_column = 256 # number of velocity channels

num_output  = 2
dropout     = 0.

# Model
cnn_trans = CNN_Transformer(pev, num_output, input_column, dropout, device)

# Load weights
# weight_cp_file = ct101_checkpoint_path / f'ctrans_{pev}_10_{input_column}_final.pth'
# cnn_trans.load_weights(weight_cp_file)

print(cnn_trans)

CNN_Transformer(
  (loss_fcn): MSELoss()
  (pos_encoder): PE()
  (conv1): Conv2d(1, 72, kernel_size=(1, 3), stride=(1, 1))
  (bn1): BatchNorm2d(72, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv2): Conv2d(72, 64, kernel_size=(1, 33), stride=(1, 1))
  (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv3): Conv2d(64, 56, kernel_size=(1, 3), stride=(1, 1))
  (bn3): BatchNorm2d(56, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv4): Conv2d(56, 48, kernel_size=(1, 33), stride=(1, 1))
  (bn4): BatchNorm2d(48, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv5): Conv2d(48, 40, kernel_size=(1, 3), stride=(1, 1))
  (bn5): BatchNorm2d(40, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv6): Conv2d(40, 32, kernel_size=(1, 33), stride=(1, 1))
  (bn6): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv7): Conv2d(32, 16, kernel_si

/usr/local/lib/python3.10/dist-packages/torch/nn/modules/transformer.py:307: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.num_heads is odd
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")


# Run

In [ ]:
cubes = [cube1, cube2]
fcnm_maps = [fcnm_data1, fcnm_data2]
rhi_maps = [rhi_data1, rhi_data2]

# data preprocessing
batch_size = 200
prep = Prep(pev, input_column, batch_size, cubes, fcnm=fcnm_maps, rhi=rhi_maps)
xtrain, ytrain = prep.fit()

# Run the training process
cnn_trans.fit(xtrain, ytrain)

# End